DATASET1 için Preprocess 

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import builtins#bu ve altındaki 3 satırda yapay zeka yardımıyla Attention modeli yükleme ve testi için önem arzediyor,Lambda kullandığımız için   
builtins.tf = tf
globals()['tf'] = tf
tf.keras.config.enable_unsafe_deserialization()
%run DeepProject.ipynb
df = pd.read_csv('usgs_main.csv')
df1 = df.copy()
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1.sort_values('time')  # Kronolojik sıralama
df1 = df1.dropna(subset=['latitude', 'longitude', 'depth', 'mag', 'time'])    
df1['lats'] = np.floor(df1['latitude']).astype(int)
df1['lons'] = np.floor(df1['longitude']).astype(int)

dfdeep = df1.set_index('time').resample('D').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean'
})
dfdeep = dfdeep.reset_index(drop=True)
dfdeep.index = dfdeep.index + 1
dfdeep.index.name = 'timeindex'
dfdeep['futuremag'] = dfdeep['mag'].shift(-1)
dfdeep['futuredepth'] = dfdeep['depth'].shift(-1)
dfdeep['futurelat'] = dfdeep['latitude'].shift(-1)
dfdeep['futurelon'] = dfdeep['longitude'].shift(-1)
dfdeep = dfdeep.dropna(subset=['futuremag', 'futuredepth', 'futurelat', 'futurelon'])

# Train-test split
train_size = int(0.8 * len(dfdeep))
train_data = dfdeep[:train_size]
test_data = dfdeep[train_size:]
def custom_softmax(x):
    return tf.nn.softmax(x, axis=1)

def custom_reduce_sum(x):
    return tf.reduce_sum(x, axis=1)

# Custom objects ile model load et
custom_objects = {
    'custom_softmax': custom_softmax,
    'custom_reduce_sum': custom_reduce_sum,
    'tf': tf  # tf referansını da ekliyoruz
}
#test için modellerimizi load ediyoruz.Böylece test verilerine  bu modelleri test ederek erişebiliriz
model = tf.keras.models.load_model('models/simpleLSTMdataset1.keras')
model2=tf.keras.models.load_model('models/LSTMAttentiondataset1.keras',safe_mode=False,custom_objects=custom_objects)
sequence_length = model.input_shape[1]
sequence_length_2=model2.input_shape[1]

LSTM deki hedef sütünları oluşturma ve sliding window oluşturma

In [3]:
#bu iki fonksiyonu koymayı önce unutmuştum,hata alınca ekledim,çünkü LSTM classlarının içindeydi.
def prepare_data(df):
    feature_cols = ['mag', 'depth', 'latitude', 'longitude']
    target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
    
    all_cols = feature_cols + target_cols
    data = df[all_cols].values
    data = data[~np.isnan(data).any(axis=1)]
    
    return data

def create_sliding_windows(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        sequence = data[i:(i + sequence_length), :4]  # features
        target = data[i + sequence_length, 4:]  # targets
        X.append(sequence)
        y.append(target)
    return np.array(X), np.array(y)


Datayı scale ettik ve train data ile test data yı oluşturduk.

In [4]:
train_data_prepared = prepare_data(train_data)

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Train verisinin feature ve target'larını ayır
train_features = train_data_prepared[:, :4]  # mag, depth, lat, lon
train_targets = train_data_prepared[:, 4:]   # future values

# Scaler'ları train verisiyle fit et
feature_scaler.fit(train_features)
target_scaler.fit(train_targets)
# 2. Test verisini hazırla
test_data_prepared = prepare_data(test_data)
test_features_scaled = feature_scaler.transform(test_data_prepared[:, :4])
test_targets_scaled = target_scaler.transform(test_data_prepared[:, 4:])
test_data_scaled = np.hstack([test_features_scaled, test_targets_scaled])

Test ve 0-1 arasına scale ettiğimiz verileri eski ölçeğine döndürme

In [5]:
X_test, y_test_scaled = create_sliding_windows(test_data_scaled, sequence_length)
test_predictions_scaled = model.predict(X_test)
test_predictions = target_scaler.inverse_transform(test_predictions_scaled)
y_test_actual = target_scaler.inverse_transform(y_test_scaled)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step


KLASİK Regresyon Metrikleri

In [6]:
min_len = min(len(test_predictions), len(y_test_actual))
print(f"\nDeğerlendirme yapılacak örnek sayısı: {min_len}")

target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']

for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions.shape[1] and i < y_test_actual.shape[1]:
        target_mse = mean_squared_error(
            y_test_actual[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_mae = mean_absolute_error(
            y_test_actual[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_r2 = r2_score(
            y_test_actual[:min_len, i], 
            test_predictions[:min_len, i]
        )
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")



Değerlendirme yapılacak örnek sayısı: 37
Magnitude   : MSE=0.063, MAE=0.214, R²=-0.938
Depth       : MSE=39.535, MAE=4.718, R²=-0.250
Latitude    : MSE=6.543, MAE=2.006, R²=-0.037
Longitude   : MSE=59.826, MAE=6.334, R²=-0.185


Threshold a göre bakma

In [7]:
magnitude_thresholds = [1.7, 2.0, 2.5, 3.0, 3.5, 4.0]
for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    # Tahmin edilen ve gerçek büyüklükler
    predicted_magnitudes = test_predictions[:min_len, 0]  
    actual_magnitudes = y_test_actual[:min_len, 0]
    # Eşiğe göre ayır
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    # Detaylı istatistikler
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()

    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    # Confusion matrix güvenli hesaplama
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except Exception as e:
        print(f"Confusion matrix hesaplanamıyor: {e}")



Büyüklük Eşiği: 1.7
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=1.7): 24
Tahmin edilen deprem sayısı (>=1.7): 0
Doğru tahmin sayısı: 13
Doğruluk (Accuracy): 0.351
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=13, FP=0, FN=24, TP=0

Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=2.0): 4
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 33
Doğruluk (Accuracy): 0.892
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=33, FP=0, FN=4, TP=0

Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 37
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 37
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

B

Konuma Göre Bakma

In [8]:
test_with_predictions = test_data.iloc[:min_len].copy()
test_with_predictions['predicted_mag'] = test_predictions[:min_len, 0]
test_with_predictions['predicted_lat'] = test_predictions[:min_len, 2]
test_with_predictions['predicted_lon'] = test_predictions[:min_len, 3]

# Daha büyük bölge grupları oluştur (1.0 derece aralıklarla)
test_with_predictions['lat_group'] = np.round(test_with_predictions['latitude'])
test_with_predictions['lon_group'] = np.round(test_with_predictions['longitude'])
test_with_predictions['location_group'] = test_with_predictions['lat_group'].astype(str) + '_' + test_with_predictions['lon_group'].astype(str)

# Her konum grubu için değerlendirme
location_groups = test_with_predictions.groupby('location_group').size()
valid_locations = location_groups[location_groups >= 1].index

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations)}")
count=0
threshold =1.6

for location in valid_locations[:10]:
    location_data = test_with_predictions[test_with_predictions['location_group'] == location]
    lat, lon = location.split('_')
    
    # threshold'dan büyük deprem var mı bakıyoruz,sondaki analiz yapay zek yardımıyla yazıldı
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    if correct_prediction:
        count+=1
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()
print(f"Doğru tahmin oranı{(count/len(valid_locations))}")
    

Yeterli veri olan bölge sayısı: 32
Bölge (32.0°, -112.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.41
  Max tahmin büyüklük: 1.61

Bölge (33.0°, -110.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.91
  Max tahmin büyüklük: 1.61

Bölge (35.0°, -114.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 2.02
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -100.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.60
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -107.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.95
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -117.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.62
  Max tahmin büyüklük: 1.61

Bölge (37.0°, -110.0°) 

Dataset1 Attention İçin İşlemleri yapalım

In [9]:
advanced_predictor = AdvancedLSTMEarthquakePredictor(
    sequence_length=sequence_length_2,
    use_outlier_detection=True,
    use_robust_scaling=True
)

# Train data preprocessing (attention model için)
train_data_preparedatt = advanced_predictor._prepare_data(train_data, is_training=True)

# Attention model için ayrı scaler'lar
feature_scaler_model2 = MinMaxScaler()
target_scaler_model2 = MinMaxScaler()

train_featuresatt = train_data_preparedatt[:, :4]
train_targetsatt = train_data_preparedatt[:, 4:] 
feature_scaler_model2.fit(train_featuresatt)
target_scaler_model2.fit(train_targetsatt)

# Test verisini hazırla (Advanced preprocessing ile)
test_data_preparedatt = advanced_predictor._prepare_data(test_data, is_training=False)
test_features_att1 = feature_scaler_model2.transform(test_data_preparedatt[:, :4])
test_targets_att1 = target_scaler_model2.transform(test_data_preparedatt[:, 4:])
test_data_att1 = np.hstack([test_features_att1, test_targets_att1])

# Sliding windows oluştur
X_test_model2, y_test_scaled_model2 = advanced_predictor._create_sliding_windows(test_data_att1)

# Tahmin yap
test_predictions_scaled_model2 = model2.predict(X_test_model2)

# Inverse transform 
test_predictions_model2 = target_scaler_model2.inverse_transform(test_predictions_scaled_model2)
y_test_actual_model2 = target_scaler_model2.inverse_transform(y_test_scaled_model2)

min_len_model2 = min(len(test_predictions_model2), len(y_test_actual_model2))

print(f"Model 2 - Değerlendirme yapılacak örnek sayısı: {min_len_model2}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
Model 2 - Değerlendirme yapılacak örnek sayısı: 38


In [10]:
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_model2.shape[1] and i < y_test_actual_model2.shape[1]:
        target_mse = mean_squared_error(
            y_test_actual_model2[:min_len_model2, i], 
            test_predictions_model2[:min_len_model2, i]
        )
        target_mae = mean_absolute_error(
            y_test_actual_model2[:min_len_model2, i], 
            test_predictions_model2[:min_len_model2, i]
        )
        target_r2 = r2_score(
            y_test_actual_model2[:min_len_model2, i], 
            test_predictions_model2[:min_len_model2, i]
        )
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

Magnitude   : MSE=0.059, MAE=0.206, R²=-0.848
Depth       : MSE=37.316, MAE=4.541, R²=-0.204
Latitude    : MSE=6.334, MAE=1.974, R²=-0.020
Longitude   : MSE=58.116, MAE=6.265, R²=-0.180


In [11]:
for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    
    predicted_magnitudes_model2 = test_predictions_model2[:min_len_model2, 0]  
    actual_magnitudes_model2 = y_test_actual_model2[:min_len_model2, 0]
    
    predicted_earthquake_model2 = (predicted_magnitudes_model2 >= threshold).astype(int)
    actual_earthquake_model2 = (actual_magnitudes_model2 >= threshold).astype(int)
    accuracy_model2 = (predicted_earthquake_model2 == actual_earthquake_model2).mean()
    
    total_samples_model2 = len(actual_earthquake_model2)
    actual_earthquakes_model2 = actual_earthquake_model2.sum()
    predicted_earthquakes_model2 = predicted_earthquake_model2.sum()
    correct_predictions_model2 = (predicted_earthquake_model2 == actual_earthquake_model2).sum()

    print(f"Toplam örnek sayısı: {total_samples_model2}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes_model2}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes_model2}")
    print(f"Doğru tahmin sayısı: {correct_predictions_model2}")
    print(f"Doğruluk (Accuracy): {accuracy_model2:.3f}")
    
    try:
        cm_model2 = confusion_matrix(actual_earthquake_model2, predicted_earthquake_model2)
        if cm_model2.size == 4:
            tn, fp, fn, tp = cm_model2.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except Exception as e:
        print(f"Confusion matrix hesaplanamıyor: {e}")


Büyüklük Eşiği: 1.7
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=1.7): 24
Tahmin edilen deprem sayısı (>=1.7): 0
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 0.368
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=14, FP=0, FN=24, TP=0

Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=2.0): 4
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 34
Doğruluk (Accuracy): 0.895
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=34, FP=0, FN=4, TP=0

Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 38
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 38
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

B

In [12]:
test_with_predictions_model2 = test_data.iloc[:min_len_model2].copy()
test_with_predictions_model2['predicted_mag'] = test_predictions_model2[:min_len_model2, 0]
test_with_predictions_model2['predicted_lat'] = test_predictions_model2[:min_len_model2, 2]
test_with_predictions_model2['predicted_lon'] = test_predictions_model2[:min_len_model2, 3]

test_with_predictions_model2['lat_group'] = np.round(test_with_predictions_model2['latitude'])
test_with_predictions_model2['lon_group'] = np.round(test_with_predictions_model2['longitude'])
test_with_predictions_model2['location_group'] = test_with_predictions_model2['lat_group'].astype(str) + '_' + test_with_predictions_model2['lon_group'].astype(str)

location_groups_model2 = test_with_predictions_model2.groupby('location_group').size()
valid_locations_model2 = location_groups_model2[location_groups_model2 >= 1].index

print(f"\nModel 2 - Yeterli veri olan bölge sayısı: {len(valid_locations_model2)}")
count_model2 = 0
threshold=1.6
print("\n=== MODEL 2 - KONUM ANALİZİ ===")
for location in valid_locations_model2[:10]:
    location_data_model2 = test_with_predictions_model2[test_with_predictions_model2['location_group'] == location]
    lat, lon = location.split('_')
    
    has_actual_earthquake_model2 = (location_data_model2['futuremag'] >= threshold).any()
    has_predicted_earthquake_model2 = (location_data_model2['predicted_mag'] >= threshold).any()
    correct_prediction_model2 = has_actual_earthquake_model2 == has_predicted_earthquake_model2
    if correct_prediction_model2:
        count_model2 += 1
        
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data_model2)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake_model2 else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake_model2 else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction_model2 else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data_model2['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data_model2['predicted_mag'].max():.2f}")
    print()

print(f"Model 2 - Doğru tahmin oranı: {(count_model2/len(valid_locations_model2)):.3f}")


Model 2 - Yeterli veri olan bölge sayısı: 33

=== MODEL 2 - KONUM ANALİZİ ===
Bölge (32.0°, -112.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.41
  Max tahmin büyüklük: 1.62

Bölge (33.0°, -110.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.91
  Max tahmin büyüklük: 1.62

Bölge (35.0°, -103.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.94
  Max tahmin büyüklük: 1.62

Bölge (35.0°, -114.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 2.02
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -100.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.60
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -107.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.95
  Max ta

DATASET2 İÇİN TESTLER,Yine preprocess adımlarını kullanıp  test ederek sonuca ulaşacağız

In [13]:
nx=pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
dfls=nx
dfls=dfls.rename(columns={'Time':'time','Mag':'mag','Depth':'depth','Latitude':'latitude','Longitude':'longitude' })
dfls['time'] = pd.to_datetime(dfls['time'])
dfls = dfls.sort_values('time')
dfls = dfls.dropna(subset=['latitude','longitude','depth','mag','time'])
dfls['lats'] = np.floor(dfls['latitude']).astype(int)
dfls['lons'] = np.floor(dfls['longitude']).astype(int)
dfother = dfls.set_index('time').resample('YE').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
})
dfother = dfother.reset_index(drop=True)
dfother.index = dfother.index + 1
dfother.index.name = 'timeindex'
dfother['futuremag']=dfother['mag'].shift(-1)
dfother['futuredepth']=dfother['depth'].shift(-1)
dfother['futurelat']=dfother['latitude'].shift(-1)
dfother['futurelon']=dfother['longitude'].shift(-1)
dfother = dfother.dropna(subset=['futuremag', 'futuredepth', 'futurelat', 'futurelon'])
train_size = int(0.8 * len(dfother))
train_data = dfother[:train_size]
test_data_2 = dfother[train_size:]
model3 = tf.keras.models.load_model('models/simpleLSTMdataset2.keras')
model4=tf.keras.models.load_model('models/LSTMAttentiondataset2.keras',safe_mode=False,custom_objects=custom_objects)
sequence_length_3 = model3.input_shape[1]
sequence_length_4=model4.input_shape[1]

aynı şekilde işliyoruz,kod bloğu üsttekinin aynısı,1 2 değişken ismi değiştirdim

In [14]:
train_data_prepared = prepare_data(train_data)
feature_scaler3 = MinMaxScaler()
target_scaler = MinMaxScaler()
# Train verisinin feature ve target'larını ayır
train_features = train_data_prepared[:, :4]  # mag, depth, lat, lon
train_targets = train_data_prepared[:, 4:]   # future values
# Scaler'ları train verisiyle fit et
feature_scaler3.fit(train_features)
target_scaler.fit(train_targets)
# 2. Test verisini hazırla
test_data_prepared = prepare_data(test_data_2)
test_features_scaled = feature_scaler3.transform(test_data_prepared[:, :4])
test_targets_scaled = target_scaler.transform(test_data_prepared[:, 4:])
test_data_scaled = np.hstack([test_features_scaled, test_targets_scaled])
X_test3, y_test_scaled = create_sliding_windows(test_data_scaled, sequence_length_3)
test_predictions_scaled = model3.predict(X_test3)
test_predictions_model3 = target_scaler.inverse_transform(test_predictions_scaled)
y_test_actual = target_scaler.inverse_transform(y_test_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step


In [15]:
min_len = min(len(test_predictions_model3), len(y_test_actual))
print(f"\nDeğerlendirme yapılacak örnek sayısı: {min_len}")

target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']

for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_model3.shape[1] and i < y_test_actual.shape[1]:
        target_mse = mean_squared_error(
            y_test_actual[:min_len, i], 
            test_predictions_model3[:min_len, i]
        )
        target_mae = mean_absolute_error(
            y_test_actual[:min_len, i], 
            test_predictions_model3[:min_len, i]
        )
        target_r2 = r2_score(
            y_test_actual[:min_len, i], 
            test_predictions_model3[:min_len, i]
        )
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")



Değerlendirme yapılacak örnek sayısı: 9
Magnitude   : MSE=0.013, MAE=0.109, R²=-17.100
Depth       : MSE=546.419, MAE=20.542, R²=-3.388
Latitude    : MSE=104.132, MAE=9.451, R²=-6.035
Longitude   : MSE=310.821, MAE=15.324, R²=-0.468


In [16]:
magnitude_thresholds = [5.8,6.0, 6.5, 7.0, 7.5]
for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    
    
    predicted_magnitudes = test_predictions_model3[:min_len, 0]  
    actual_magnitudes = y_test_actual[:min_len, 0]
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 5.8
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=5.8): 9
Tahmin edilen deprem sayısı (>=5.8): 0
Doğru tahmin sayısı: 0
Doğruluk (Accuracy): 0.000
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=0, FP=0, FN=9, TP=0

Büyüklük Eşiği: 6.0
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=6.0): 0
Tahmin edilen deprem sayısı (>=6.0): 0
Doğru tahmin sayısı: 9
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.5
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=6.5): 0
Tahmin edilen deprem sayısı (>=6.5): 0
Doğru tahmin sayısı: 9
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.0
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=7.0): 0
Tahmin edilen deprem sayısı (>=7.0): 0
Doğru tahmin sayısı: 9
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.5
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=7.5): 0

In [17]:
test_with_predictions_2 = test_data_2.iloc[:min_len].copy()
test_with_predictions_2['predicted_mag'] = test_predictions_model3[:min_len, 0]
test_with_predictions_2['predicted_lat'] = test_predictions_model3[:min_len, 2]
test_with_predictions_2['predicted_lon'] = test_predictions_model3[:min_len, 3]


test_with_predictions_2['lat_group'] = np.round(test_with_predictions_2['latitude'] / 5.0) * 5.0
test_with_predictions_2['lon_group'] = np.round(test_with_predictions_2['longitude'] / 5.0) * 5.0
test_with_predictions_2['location_group'] = test_with_predictions_2['lat_group'].astype(str) + '_' + test_with_predictions_2['lon_group'].astype(str)


location_groups_2 = test_with_predictions_2.groupby('location_group').size()
valid_locations_2 = location_groups_2[location_groups_2 >= 1].index 

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_2)}")

threshold = 5.9 
count=0
for location in valid_locations_2[:10]:
    location_data = test_with_predictions_2[test_with_predictions_2['location_group'] == location]
    lat, lon = location.split('_')
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    if correct_prediction:
        count+=1
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()
print(f"accuracy: {count/len(valid_locations_2)}")

Yeterli veri olan bölge sayısı: 8
Bölge (-0.0°, 25.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.87
  Max tahmin büyüklük: 5.77

Bölge (-0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.77

Bölge (-0.0°, 55.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.88
  Max tahmin büyüklük: 5.77

Bölge (-5.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.91
  Max tahmin büyüklük: 5.77

Bölge (0.0°, 40.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.89
  Max tahmin büyüklük: 5.77

Bölge (0.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.77

Bölge (5.0°, 40.0°) - Veri sayısı: 1
 

Son Olarak DATASET2 ATTENTIONLI MEKANİZMA

In [18]:
advanced_predictor = AdvancedLSTMEarthquakePredictor(
    sequence_length=sequence_length_4,
    use_outlier_detection=True,
    use_robust_scaling=True
)

# Train data preprocessing (attention model için)
train_data_preparedatt = advanced_predictor._prepare_data(train_data, is_training=True)

# Attention model için ayrı scaler'lar
feature_scaler_model4 = MinMaxScaler()
target_scaler_model4 = MinMaxScaler()

train_featuresatt = train_data_preparedatt[:, :4]
train_targetsatt = train_data_preparedatt[:, 4:] 
feature_scaler_model4.fit(train_featuresatt)
target_scaler_model4.fit(train_targetsatt)

# Test verisini hazırla (Advanced preprocessing ile)
test_data_preparedatt = advanced_predictor._prepare_data(test_data_2, is_training=False)
test_features_att2 = feature_scaler_model4.transform(test_data_preparedatt[:, :4])
test_targets_att2 = target_scaler_model4.transform(test_data_preparedatt[:, 4:])
test_data_att2 = np.hstack([test_features_att2, test_targets_att2])

# Sliding windows oluştur
X_test_model4, y_test_scaled_model4 = advanced_predictor._create_sliding_windows(test_data_att2)

# Tahmin yap
test_predictions_scaled_model2 = model4.predict(X_test_model4)

# Inverse transform 
test_predictions_model4 = target_scaler_model4.inverse_transform(test_predictions_scaled_model2)
y_test_actual_model2 = target_scaler_model4.inverse_transform(y_test_scaled_model4)

min_len_model4 = min(len(test_predictions_model4), len(y_test_actual_model2))

print(f"Model 4 - Değerlendirme yapılacak örnek sayısı: {min_len_model4}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
Model 4 - Değerlendirme yapılacak örnek sayısı: 14


In [19]:
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_model4.shape[1] and i < y_test_actual_model2.shape[1]:
        target_mse = mean_squared_error(
            y_test_actual_model2[:min_len_model4, i], 
            test_predictions_model4[:min_len_model4, i]
        )
        target_mae = mean_absolute_error(
            y_test_actual_model2[:min_len_model4, i], 
            test_predictions_model4[:min_len_model4, i]
        )
        target_r2 = r2_score(
            y_test_actual_model2[:min_len_model4, i], 
            test_predictions_model4[:min_len_model4, i]
        )
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

Magnitude   : MSE=0.004, MAE=0.061, R²=-7.222
Depth       : MSE=780.824, MAE=26.000, R²=-6.713
Latitude    : MSE=26.011, MAE=4.343, R²=-0.150
Longitude   : MSE=920.628, MAE=25.824, R²=-2.556


In [20]:
for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    
    predicted_magnitudes_model2 = test_predictions_model4[:min_len_model4, 0]  
    actual_magnitudes_model2 = y_test_actual_model2[:min_len_model4, 0]
    
    predicted_earthquake_model2 = (predicted_magnitudes_model2 >= threshold).astype(int)
    actual_earthquake_model2 = (actual_magnitudes_model2 >= threshold).astype(int)
    accuracy_model2 = (predicted_earthquake_model2 == actual_earthquake_model2).mean()
    
    total_samples_model2 = len(actual_earthquake_model2)
    actual_earthquakes_model2 = actual_earthquake_model2.sum()
    predicted_earthquakes_model2 = predicted_earthquake_model2.sum()
    correct_predictions_model4 = (predicted_earthquake_model2 == actual_earthquake_model2).sum()

    print(f"Toplam örnek sayısı: {total_samples_model2}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes_model2}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes_model2}")
    print(f"Doğru tahmin sayısı: {correct_predictions_model4}")
    print(f"Doğruluk (Accuracy): {accuracy_model2:.3f}")
    
    try:
        cm_model2 = confusion_matrix(actual_earthquake_model2, predicted_earthquake_model2)
        if cm_model2.size == 4:
            tn, fp, fn, tp = cm_model2.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except Exception as e:
        print(f"Confusion matrix hesaplanamıyor: {e}")


Büyüklük Eşiği: 5.8
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=5.8): 14
Tahmin edilen deprem sayısı (>=5.8): 14
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.0
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=6.0): 0
Tahmin edilen deprem sayısı (>=6.0): 0
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.5
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=6.5): 0
Tahmin edilen deprem sayısı (>=6.5): 0
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.0
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=7.0): 0
Tahmin edilen deprem sayısı (>=7.0): 0
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.5
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=7.5): 0
Tahmin edilen deprem sayısı (>=7.5): 0
Doğru tahm

In [21]:
test_with_predictions_2 = test_data_2.iloc[:min_len_model4].copy()
test_with_predictions_2['predicted_mag'] = test_predictions_model4[:min_len_model4, 0]
test_with_predictions_2['predicted_lat'] = test_predictions_model4[:min_len_model4, 2]
test_with_predictions_2['predicted_lon'] = test_predictions_model4[:min_len_model4, 3]


test_with_predictions_2['lat_group'] = np.round(test_with_predictions_2['latitude'] / 5.0) * 5.0
test_with_predictions_2['lon_group'] = np.round(test_with_predictions_2['longitude'] / 5.0) * 5.0
test_with_predictions_2['location_group'] = test_with_predictions_2['lat_group'].astype(str) + '_' + test_with_predictions_2['lon_group'].astype(str)


location_groups_2 = test_with_predictions_2.groupby('location_group').size()
valid_locations_2 = location_groups_2[location_groups_2 >= 1].index 

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_2)}")

threshold = 5.9 
count=0
for location in valid_locations_2[:10]:
    location_data = test_with_predictions_2[test_with_predictions_2['location_group'] == location]
    lat, lon = location.split('_')
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    if correct_prediction:
        count+=1
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()
print(f"accuracy: {count/len(valid_locations_2)}")

Yeterli veri olan bölge sayısı: 12
Bölge (-0.0°, 25.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.87
  Max tahmin büyüklük: 5.94

Bölge (-0.0°, 30.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.90
  Max tahmin büyüklük: 5.94

Bölge (-0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.94

Bölge (-0.0°, 55.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.88
  Max tahmin büyüklük: 5.94

Bölge (-5.0°, 20.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.94

Bölge (-5.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.91
  Max tahmin büyüklük: 5.94

Bölge (0.0°, 40.0°) - Veri sayısı: 

TEST VERİSİNDEN ÖRNEK VE TAHMİNİ

In [22]:
# Tek bir veri örneği için tahmin yapma ve gerçek değerle karşılaştırma,basit bir temel mantığı var
# ilk model için yaptıktan sonra kalanları hızlıca değişken isimlerini değiştirerek doldurdum 
#rastgelelik için index eklemek yapay zeka önerisiydi
print("DATASET 1 - USGS DATA")
# Test verisinden rastgele bir örnek seç
sample_index = 5  # İstediğiniz index'i buraya yazabilirsiniz
print(f"\nSeçilen örnek index: {sample_index}")

if sample_index < len(test_data):
    # Gerçek değerleri al
    sample_row = test_data.iloc[sample_index]
    
    print("\n--- GERİ GERÇEK DEĞERLER ---")
    print(f"Mevcut Büyüklük (mag): {sample_row['mag']:.3f}")
    print(f"Mevcut Derinlik (depth): {sample_row['depth']:.3f}")
    print(f"Mevcut Enlem (latitude): {sample_row['latitude']:.3f}")
    print(f"Mevcut Boylam (longitude): {sample_row['longitude']:.3f}")
    
    print(f"\nGelecek Büyüklük (futuremag): {sample_row['futuremag']:.3f}")
    print(f"Gelecek Derinlik (futuredepth): {sample_row['futuredepth']:.3f}")
    print(f"Gelecek Enlem (futurelat): {sample_row['futurelat']:.3f}")
    print(f"Gelecek Boylam (futurelon): {sample_row['futurelon']:.3f}")
    
    # Model 1 (Simple LSTM) tahminleri
    if sample_index < len(test_predictions):
        print("\nMODEL 1")
        pred_mag = test_predictions[sample_index, 0]
        pred_depth = test_predictions[sample_index, 1]
        pred_lat = test_predictions[sample_index, 2]
        pred_lon = test_predictions[sample_index, 3]
        
        print(f"Tahmin Büyüklük: {pred_mag:.3f}")
        print(f"Tahmin Derinlik: {pred_depth:.3f}")
        print(f"Tahmin Enlem: {pred_lat:.3f}")
        print(f"Tahmin Boylam: {pred_lon:.3f}")
        
        # Hatalar
       
        mag_error = abs(sample_row['futuremag'] - pred_mag)
        depth_error = abs(sample_row['futuredepth'] - pred_depth)
        lat_error = abs(sample_row['futurelat'] - pred_lat)
        lon_error = abs(sample_row['futurelon'] - pred_lon)
        
        print(f"Büyüklük hatası: {mag_error:.3f}")
        print(f"Derinlik hatası: {depth_error:.3f}")
        print(f"Enlem hatası: {lat_error:.3f}")
        print(f"Boylam hatası: {lon_error:.3f}")
        
    # Model 2 (LSTM + Attention) tahminleri
    if sample_index < len(test_predictions_model2):
        print("\n MODEL 2")
        pred_mag_2 = test_predictions_model2[sample_index, 0]
        pred_depth_2 = test_predictions_model2[sample_index, 1]
        pred_lat_2 = test_predictions_model2[sample_index, 2]
        pred_lon_2 = test_predictions_model2[sample_index, 3]
        
        print(f"Tahmin Büyüklük: {pred_mag_2:.3f}")
        print(f"Tahmin Derinlik: {pred_depth_2:.3f}")
        print(f"Tahmin Enlem: {pred_lat_2:.3f}")
        print(f"Tahmin Boylam: {pred_lon_2:.3f}")
        
        # Hatalar
        
        
        mag_error_2 = abs(sample_row['futuremag'] - pred_mag_2)
        depth_error_2 = abs(sample_row['futuredepth'] - pred_depth_2)
        lat_error_2 = abs(sample_row['futurelat'] - pred_lat_2)
        lon_error_2 = abs(sample_row['futurelon'] - pred_lon_2)
        
        print(f"Büyüklük hatası: {mag_error_2:.3f}")
        print(f"Derinlik hatası: {depth_error_2:.3f}")
        print(f"Enlem hatası: {lat_error_2:.3f}")
        print(f"Boylam hatası: {lon_error_2:.3f}")
        # Model karşılaştırması
        if sample_index < len(test_predictions):
            print(f"\n--- MODEL KARŞILAŞTIRMASI ---")
            print(f"Model 1 büyüklük hatası: {mag_error:.3f}")
            print(f"Model 2 büyüklük hatası: {mag_error_2:.3f}")
            better_model = "Model 2" if mag_error_2 < mag_error else "Model 1"
            print(f"Bu örnek için daha iyi: {better_model}")


print("DATASET 2 - SIGNIFICANT EARTHQUAKES")


# Dataset 2 test verisi mevcut mu kontrol et
if 'dfother' in locals() and len(dfother) > 0:
    # Train-test split Dataset 2 için
    train_size_2 = int(0.8 * len(dfother))
    test_data_2 = dfother[train_size_2:]
    
    sample_index_2 = 2  # Dataset 2 için örnek index
    print(f"\nSeçilen örnek index: {sample_index_2}")
    
    if sample_index_2 < len(test_data_2):
        sample_row_2 = test_data_2.iloc[sample_index_2]
        
        print("\n--- GERÇEK DEĞERLER ---")
        print(f"Mevcut Büyüklük (mag): {sample_row_2['mag']:.3f}")
        print(f"Mevcut Derinlik (depth): {sample_row_2['depth']:.3f}")
        print(f"Mevcut Enlem (latitude): {sample_row_2['latitude']:.3f}")
        print(f"Mevcut Boylam (longitude): {sample_row_2['longitude']:.3f}")
        
        print(f"\nGelecek Büyüklük (futuremag): {sample_row_2['futuremag']:.3f}")
        print(f"Gelecek Derinlik (futuredepth): {sample_row_2['futuredepth']:.3f}")
        print(f"Gelecek Enlem (futurelat): {sample_row_2['futurelat']:.3f}")
        print(f"Gelecek Boylam (futurelon): {sample_row_2['futurelon']:.3f}")
if sample_index_2 < len(test_predictions_model3):
            print("\nModel 3")
            pred_mag_3 = test_predictions_model3[sample_index_2, 0]
            pred_depth_3 = test_predictions_model3[sample_index_2, 1]
            pred_lat_3 = test_predictions_model3[sample_index_2, 2]
            pred_lon_3 = test_predictions_model3[sample_index_2, 3]
            print(f"Tahmin Büyüklük: {pred_mag_3:.3f}")
            print(f"Tahmin Derinlik: {pred_depth_3:.3f}")
            print(f"Tahmin Enlem: {pred_lat_3:.3f}")
            print(f"Tahmin Boylam: {pred_lon_3:.3f}")

            mag_error_3 = abs(sample_row_2['futuremag'] - pred_mag_3)
            depth_error_3 = abs(sample_row_2['futuredepth'] - pred_depth_3)
            lat_error_3 = abs(sample_row_2['futurelat'] - pred_lat_3)
            lon_error_3 = abs(sample_row_2['futurelon'] - pred_lon_3)
            print(f"Büyüklük hatası: {mag_error_3:.3f}")
            print(f"Derinlik hatası: {depth_error_3:.3f}")
            print(f"Enlem hatası: {lat_error_3:.3f}")
            print(f"Boylam hatası: {lon_error_3:.3f}")
            print("\n Model 4")
            pred_mag_4 = test_predictions_model4[sample_index_2, 0]
            pred_depth_4 = test_predictions_model4[sample_index_2, 1]
            pred_lat_4 = test_predictions_model4[sample_index_2, 2]
            pred_lon_4 = test_predictions_model4[sample_index_2, 3]
            
            print(f"Tahmin Büyüklük: {pred_mag_4:.3f}")
            print(f"Tahmin Derinlik: {pred_depth_4:.3f}")
            print(f"Tahmin Enlem: {pred_lat_4:.3f}")
            print(f"Tahmin Boylam: {pred_lon_4:.3f}")
            
            
            mag_error_4 = abs(sample_row_2['futuremag'] - pred_mag_4)
            depth_error_4 = abs(sample_row_2['futuredepth'] - pred_depth_4)
            lat_error_4 = abs(sample_row_2['futurelat'] - pred_lat_4)
            lon_error_4 = abs(sample_row_2['futurelon'] - pred_lon_4)
            
            print(f"Büyüklük hatası: {mag_error_4:.3f}")
            print(f"Derinlik hatası: {depth_error_4:.3f}")
            print(f"Enlem hatası: {lat_error_4:.3f}")
            print(f"Boylam hatası: {lon_error_4:.3f}")
            
            if sample_index < len(test_predictions):
             print(f"\n--- MODEL KARŞILAŞTIRMASI ---")
             print(f"Model 3 büyüklük hatası: {mag_error_3:.3f}")
             print(f"Model 4 büyüklük hatası: {mag_error_4:.3f}")
             better_model = "Model 4" if mag_error_4 < mag_error_3 else "Model 3"
             print(f"Bu örnek için daha iyi: {better_model}")



DATASET 1 - USGS DATA

Seçilen örnek index: 5

--- GERİ GERÇEK DEĞERLER ---
Mevcut Büyüklük (mag): 1.723
Mevcut Derinlik (depth): 20.683
Mevcut Enlem (latitude): 38.184
Mevcut Boylam (longitude): -112.003

Gelecek Büyüklük (futuremag): 1.945
Gelecek Derinlik (futuredepth): 24.520
Gelecek Enlem (futurelat): 36.985
Gelecek Boylam (futurelon): -110.187

MODEL 1
Tahmin Büyüklük: 1.612
Tahmin Derinlik: 20.460
Tahmin Enlem: 37.381
Tahmin Boylam: -111.720
Büyüklük hatası: 0.334
Derinlik hatası: 4.061
Enlem hatası: 0.396
Boylam hatası: 1.533

 MODEL 2
Tahmin Büyüklük: 1.621
Tahmin Derinlik: 20.626
Tahmin Enlem: 37.408
Tahmin Boylam: -111.652
Büyüklük hatası: 0.325
Derinlik hatası: 3.895
Enlem hatası: 0.423
Boylam hatası: 1.465

--- MODEL KARŞILAŞTIRMASI ---
Model 1 büyüklük hatası: 0.334
Model 2 büyüklük hatası: 0.325
Bu örnek için daha iyi: Model 2
DATASET 2 - SIGNIFICANT EARTHQUAKES

Seçilen örnek index: 2

--- GERÇEK DEĞERLER ---
Mevcut Büyüklük (mag): 5.878
Mevcut Derinlik (depth): 66.420


KLASÖR OKUMA

In [24]:
#bu kod bloğunda yapay zekadan çokça faydalanıldı
import os
import glob

def create_sample_folder_fixed(test_data, folder_name="demo_samples_fixed", n_samples=25, sequence_length=10):
    #Test verisinden sequence_length+1 kadar ardışık satır alıp klasöre kaydeder
    # Klasör oluştur
    os.makedirs(folder_name, exist_ok=True)
    
    print(f"{folder_name} klasörü oluşturuluyor...")
    
    # Test verisinin uzunluğunu kontrol et
    max_start_idx = len(test_data) - sequence_length - 1
    # Rastgele başlangıç noktaları seçme
    np.random.seed(42)
    sample_count = min(n_samples, max_start_idx)
    start_indices = np.random.choice(max_start_idx, size=sample_count, replace=False)
    
    for i, start_idx in enumerate(start_indices):#bu for döngüsünü yapay zeka ekledi,oldukça anlaşılır formatta
        # sequence_length + 1 kadar ardışık satır al
        end_idx = start_idx + sequence_length + 1
        sample_chunk = test_data.iloc[start_idx:end_idx]
        
        filename = f"{folder_name}/sample_{i+1:02d}.csv"
        sample_chunk.to_csv(filename, index=False)
    
    print(f"{len(start_indices)} adet örnek {folder_name} klasörüne kaydedildi.")
    print(f"Her dosya {sequence_length + 1} satır içeriyor.")
    return folder_name


def predict_from_folder_fixed(folder_name, model, feature_scaler, target_scaler, sequence_length):
    #Klasördeki tüm CSV dosyalarını okuyup tahmin yapar
    
    csv_files = glob.glob(f"{folder_name}/*.csv")
    csv_files.sort()
    
    all_predictions = []
    all_actuals = []
    all_filenames = []
    
    print(f"\n{folder_name} klasöründeki {len(csv_files)} dosya işleniyor...")
    
    for csv_file in csv_files:
        try:
            # Dosyayı oku
            sample_df = pd.read_csv(csv_file)
            print(f"{os.path.basename(csv_file)}: {len(sample_df)} satır okundu")
            
            # Minimum satır kontrolü,yapay zekanın eklediği debug
            if len(sample_df) < sequence_length + 1:
                continue
            
            # Veriyi hazırla
            sample_prepared = prepare_data(sample_df)
            
            if len(sample_prepared) < sequence_length + 1:#hazırlanan veri yeterli mi
                continue
            
            # Scaling ve sliding window oluşturma
            sample_features_scaled = feature_scaler.transform(sample_prepared[:, :4])
            sample_targets_scaled = target_scaler.transform(sample_prepared[:, 4:])
            sample_data_scaled = np.hstack([sample_features_scaled, sample_targets_scaled])
            
            X_sample, y_sample = create_sliding_windows(sample_data_scaled, sequence_length)
            
            if len(X_sample) == 0:
                print(f"{csv_file}: Sliding window oluşturulamadı")
                continue
            
            # Tahmin yap
            pred_scaled = model.predict(X_sample, verbose=0)
            prediction = target_scaler.inverse_transform(pred_scaled)
            actual = target_scaler.inverse_transform(y_sample)
            
            # Sonuçları kaydet
            all_predictions.extend(prediction)
            all_actuals.extend(actual)
            all_filenames.extend([os.path.basename(csv_file)] * len(prediction))
            #exception ı yapay zeka ekledi
            print(f"{os.path.basename(csv_file)}: {len(prediction)} tahmin yapıldı")
            
        except Exception as e:
            print(f"{csv_file}: Hata - {str(e)}")
    
    return np.array(all_predictions), np.array(all_actuals), all_filenames

# 3. Ana demo fonksiyonu
def run_demo_fixed(test_data, model, feature_scaler, target_scaler, sequence_length, 
                   model_name="Model", n_samples=25):
    #Demoyu çalıştırır
    # Klasör oluştur ve örnekleri kaydet(AI)
    folder_name = f"demo_samples_fixed_{model_name.lower().replace(' ', '_')}"
    create_sample_folder_fixed(test_data, folder_name, n_samples, sequence_length)
    
    predictions, actuals, filenames = predict_from_folder_fixed(
        folder_name, model, feature_scaler, target_scaler, sequence_length
    )
    
    
    
    return predictions, actuals, filenames

# 4. Tek dosya için tahmin
def predict_single_file_fixed(file_path, model, feature_scaler, target_scaler, sequence_length):
    
    
    print(f"\nTEK DOSYA TAHMİNİ: {os.path.basename(file_path)}")
    print("="*60)
    
    try:
        # Dosyayı oku
        sample_df = pd.read_csv(file_path)
        
        # Minimum satır kontrolü
        if len(sample_df) < sequence_length + 1:
            return None, None
        # Veriyi hazırla
        sample_prepared = prepare_data(sample_df)
        
        if len(sample_prepared) < sequence_length + 1:
            return None, None
        
        # Scale et
        sample_features_scaled = feature_scaler.transform(sample_prepared[:, :4])
        sample_targets_scaled = target_scaler.transform(sample_prepared[:, 4:])
        sample_data_scaled = np.hstack([sample_features_scaled, sample_targets_scaled])
        
        # Sliding window oluştur
        X_sample, y_sample = create_sliding_windows(sample_data_scaled, sequence_length)
        
        if len(X_sample) == 0:#sliding window oluşmadıysa
            return None, None
        
        print(f"{len(X_sample)} sliding window oluşturuldu")
        
        # Tahmin yap
        pred_scaled = model.predict(X_sample, verbose=0)
        prediction = target_scaler.inverse_transform(pred_scaled)
        actual = target_scaler.inverse_transform(y_sample)
        # Sonuçları göster
    
        target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
        
        # İlk 2 tahmini göster
        show_count = min(2, len(prediction))
        for i in range(show_count):#AI yazdı
            print(f"\nTahmin {i+1}:")
            for j, label in enumerate(target_labels):
                if j < prediction.shape[1] and j < actual.shape[1]:
                    error = abs(actual[i, j] - prediction[i, j])
                    print(f"  {label:<12}: Gerçek={actual[i, j]:.4f}, "
                          f"Tahmin={prediction[i, j]:.4f}, Hata={error:.4f}")
        
        if len(prediction) > 2:
            print(f"\n... (toplam {len(prediction)} tahmin)")
        
        # Ortalama metrikler
        print(f"\nORTALAMA PERFORMANS:")
        print("-"*30)
        for j, label in enumerate(target_labels):
            if j < prediction.shape[1] and j < actual.shape[1]:
                mse = mean_squared_error(actual[:, j], prediction[:, j])
                mae = mean_absolute_error(actual[:, j], prediction[:, j])
                print(f"  {label:<12}: MSE={mse:.4f}, MAE={mae:.4f}")
        
        return prediction, actual
        
    except Exception as e:
        print(f" Hata: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

# DATASET 1 için demo
# Model 1 (Simple LSTM) demo
pred1, act1, files1 = run_demo_fixed(
    test_data, model, feature_scaler, target_scaler, 
    sequence_length, "Model_1_Simple_LSTM", 15  # Daha az örnek
)
# İlk oluşturulan klasörden bir dosya seç
print("ilk klasörden dosya seç")
sample_file = "demo_samples_fixed_model_1_simple_lstm/sample_01.csv"
if os.path.exists(sample_file):
    pred_single, act_single = predict_single_file_fixed(
        sample_file, model, feature_scaler, target_scaler, sequence_length
    )
else:
    print(f"Örnek dosya bulunamadı: {sample_file}")



demo_samples_fixed_model_1_simple_lstm klasörü oluşturuluyor...
15 adet örnek demo_samples_fixed_model_1_simple_lstm klasörüne kaydedildi.
Her dosya 21 satır içeriyor.

demo_samples_fixed_model_1_simple_lstm klasöründeki 15 dosya işleniyor...
sample_01.csv: 21 satır okundu
sample_01.csv: 1 tahmin yapıldı
sample_02.csv: 21 satır okundu
sample_02.csv: 1 tahmin yapıldı
sample_03.csv: 21 satır okundu
sample_03.csv: 1 tahmin yapıldı
sample_04.csv: 21 satır okundu
sample_04.csv: 1 tahmin yapıldı
sample_05.csv: 21 satır okundu
sample_05.csv: 1 tahmin yapıldı
sample_06.csv: 21 satır okundu
sample_06.csv: 1 tahmin yapıldı
sample_07.csv: 21 satır okundu
sample_07.csv: 1 tahmin yapıldı
sample_08.csv: 21 satır okundu
sample_08.csv: 1 tahmin yapıldı
sample_09.csv: 21 satır okundu
sample_09.csv: 1 tahmin yapıldı
sample_10.csv: 21 satır okundu
sample_10.csv: 1 tahmin yapıldı
sample_11.csv: 21 satır okundu
sample_11.csv: 1 tahmin yapıldı
sample_12.csv: 21 satır okundu
sample_12.csv: 1 tahmin yapıldı
s